In [ ]:
# 🎯 Objective:
# Build a multi-step intelligent advisor system using agent workflows.

# 💼 Problem Statement
# Students need guidance on career paths based on their skills.

# Your system should:

# Analyze student profile
# Suggest career roles
# Generate learning roadmap
# 👥 Agents to Build
# Profile Analyzer Agent
# Understands student background
# Career Recommendation Agent
# Suggests suitable roles
# Skill Gap Agent
# Identifies missing skills
# Learning Path Agent
# Creates step-by-step roadmap
# ⚙️ Tasks to Implement
# Sequential workflow (LangGraph preferred)
# State passing between agents
# Conditional logic (beginner vs intermediate)
# 💡 Sample Input
# “I know Python basics and statistics, and I want to become a Data Scientist”

# 🧪 Expected Output
# Suggested roles (e.g., Data Scientist, Analyst)
# Skill gaps (ML, SQL, etc.)
# Learning roadmap (step-by-step)
# 🧰 Tools (Choose Any)
# LangGraph (recommended)
# CrewAI
# AutoGen
# Python logic-based workflow
# Microsoft Copilot Studio
# 📦 Deliverables
# Workflow implementation
# Output for at least 2 student profiles
# Diagram of agent flow
# use langraph

In [ ]:
# Workflow Diagram

#                 ┌──────────────────────┐
# Student Input ─►│ Profile Analyzer     │
#                 └─────────┬────────────┘
#                           ▼
#                 ┌──────────────────────┐
#                 │ Career Recommendation│
#                 └─────────┬────────────┘
#                           ▼
#                 ┌──────────────────────┐
#                 │ Skill Gap Analyzer   │
#                 └─────────┬────────────┘
#                           ▼
#                 ┌──────────────────────┐
#                 │ Learning Path Agent  │
#                 └─────────┬────────────┘
#                           ▼
#                        Final Output

In [3]:
import os
from typing import TypedDict
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

load_dotenv()

# NVIDIA LLM
llm = ChatOpenAI(
    model="meta/llama3-70b-instruct",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0
)

# ---------------- STATE ---------------- #

class CareerState(TypedDict):
    profile: str
    level: str
    roles: str
    skill_gaps: str
    roadmap: str


# ---------------- AGENTS ---------------- #

def profile_analyzer(state: CareerState):

    prompt = f"""
    Analyze student profile and determine level:
    beginner
    intermediate

    Profile:
    {state['profile']}

    Output only one word.
    """

    response = llm.invoke(prompt).content

    state["level"] = response.strip().lower()

    return state


def career_agent(state: CareerState):

    prompt = f"""
    Based on profile suggest 2-3 career roles.

    Profile:
    {state['profile']}

    Keep answer concise.
    """

    response = llm.invoke(prompt).content

    state["roles"] = response

    return state


def skill_gap_agent(state: CareerState):

    prompt = f"""
    Identify only 5 key missing skills.

    Roles:
    {state['roles']}

    Profile:
    {state['profile']}

    Return short bullet points only.
    """

    response = llm.invoke(prompt).content

    state["skill_gaps"] = response

    return state


def learning_path_agent(state: CareerState):

    if "beginner" in state["level"]:
        level_instruction = "Create 6 step beginner roadmap"
    else:
        level_instruction = "Create 6 step intermediate roadmap"

    prompt = f"""
    {level_instruction}

    Skill gaps:
    {state['skill_gaps']}

    Keep each step short (max 8 words).
    """

    response = llm.invoke(prompt).content

    state["roadmap"] = response

    return state


def final_formatter(state: CareerState):

    final_output = f"""
Suggested Roles:
{state['roles']}

Skill Gaps:
{state['skill_gaps']}

Learning Roadmap:
{state['roadmap']}
"""

    state["roadmap"] = final_output

    return state


# ---------------- GRAPH ---------------- #

builder = StateGraph(CareerState)

builder.add_node("profile", profile_analyzer)

builder.add_node("career", career_agent)

builder.add_node("skills", skill_gap_agent)

builder.add_node("roadmap", learning_path_agent)

builder.add_node("final", final_formatter)

# sequential flow

builder.set_entry_point("profile")

builder.add_edge("profile", "career")

builder.add_edge("career", "skills")

builder.add_edge("skills", "roadmap")

builder.add_edge("roadmap", "final")

builder.add_edge("final", END)

graph = builder.compile()


# ---------------- TEST ---------------- #

profiles = [

"I know Python basics and statistics, and I want to become a Data Scientist",

"I am good at HTML CSS JavaScript and React, want to become Frontend Developer"

]

for p in profiles:

    result = graph.invoke({
        "profile": p
    })

    print("\n============================")
    print(result["roadmap"])



Suggested Roles:
Based on your profile, here are 2-3 career roles that may be a good fit:

1. **Data Analyst**: You can start by working as a data analyst, where you'll apply your Python and statistics skills to analyze and interpret data.
2. **Junior Data Scientist**: With your foundation in Python and statistics, you can take on a junior data scientist role, where you'll work on data modeling, visualization, and machine learning projects.
3. **Business Intelligence Developer**: In this role, you'll use your Python skills to develop data visualizations and reports, and apply statistical knowledge to inform business decisions.

Skill Gaps:
Here are 5 key missing skills:

• Data modeling
• Data visualization
• Machine learning
• Business acumen
• Data storytelling

Learning Roadmap:
Here is a 6-step intermediate roadmap to address the skill gaps:

**Step 1: Learn Data Modeling Fundamentals Online**

**Step 2: Practice Data Visualization with Tableau**

**Step 3: Take Machine Learning 